In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

# Forzar el uso de todos los hilos del procesador
tf.config.threading.set_intra_op_parallelism_threads(12)
tf.config.threading.set_inter_op_parallelism_threads(12)

# Configurar el crecimiento de memoria inmediatamente antes de cualquier otra cosa
gpus = tf.config.list_physical_devices('GPU')
print(f"GPUs detectadas: {len(gpus)}")
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("Crecimiento de memoria de la GPU activado con éxito.")
    except RuntimeError as e:
        print("Aviso: La GPU ya había sido inicializada:", e)

# ==========================================
# 1. Parámetros y Solución Original (Ground Truth)
# ==========================================
lam = 0.05  # Perturbación de Duffing

def get_ground_truth(t_max, x_points, t_points):
    def duffing_wkb_odes(t, y):
        a0, g0, d1, a1, g1 = y
        return [
            1j*(2*a0**2 - 0.5), 1j*a0,
            1j*(8*a0*d1 - 1), 1j*(4*a0*a1 - 6*d1), 1j*a1
        ]

    y0 = [0.5 + 0j, 0.25*np.log(np.pi) + 0j, 0j, 0j, 0j]
    t_eval = np.linspace(0, t_max, t_points)
    sol = solve_ivp(duffing_wkb_odes, [0, t_max], y0, t_eval=t_eval, method='RK45')

    a0, g0, d1, a1, g1 = sol.y
    x = np.linspace(-3, 3, x_points)

    W_exact = np.zeros((t_points, x_points))
    for i in range(t_points):
        S0 = a0[i]*x**2 + g0[i]
        S1 = a1[i]*x**2 + d1[i]*x**4 + g1[i]
        S_tot = S0 + lam * S1
        psi = np.exp(-S_tot)
        W_exact[i] = np.abs(psi)**2
        W_exact[i] /= np.sum(W_exact[i]) * (x[1]-x[0])

    return x, t_eval, W_exact

# ==========================================
# 2. Arquitectura de la PINN y Física (TensorFlow)
# ==========================================
class RiccatiPINN(tf.keras.Model):
    def __init__(self):
        super(RiccatiPINN, self).__init__()

        act_fn = tf.nn.swish
        init = tf.keras.initializers.GlorotNormal()

        self.d1 = tf.keras.layers.Dense(128, activation=act_fn, kernel_initializer=init)
        self.d2 = tf.keras.layers.Dense(128, activation=act_fn, kernel_initializer=init)
        self.d3 = tf.keras.layers.Dense(128, activation=act_fn, kernel_initializer=init)
        self.d4 = tf.keras.layers.Dense(128, activation=act_fn, kernel_initializer=init)
        self.d5 = tf.keras.layers.Dense(128, activation=act_fn, kernel_initializer=init)

        self.out = tf.keras.layers.Dense(2, kernel_initializer=init)

    def call(self, inputs):
        z = self.d1(inputs)
        z = self.d2(z)
        z = self.d3(z)
        z = self.d4(z)
        z = self.d5(z)
        return self.out(z)

def physics_loss(model, x, t):
    with tf.GradientTape(persistent=True) as tape2:
        tape2.watch(x)
        with tf.GradientTape(persistent=True) as tape1:
            tape1.watch([x, t])
            inputs = tf.concat([x, t], axis=1)
            S = model(inputs)
            S_R = S[:, 0:1]
            S_I = S[:, 1:2]

        S_R_x = tape1.gradient(S_R, x)
        S_I_x = tape1.gradient(S_I, x)
        S_R_t = tape1.gradient(S_R, t)
        S_I_t = tape1.gradient(S_I, t)

    S_R_xx = tape2.gradient(S_R_x, x)
    S_I_xx = tape2.gradient(S_I_x, x)

    del tape1, tape2

    V = 0.5 * x**2 + lam * x**4

    f_R = S_I_t - 0.5*S_R_xx + 0.5*S_R_x**2 - 0.5*S_I_x**2 - V
    f_I = -S_R_t - 0.5*S_I_xx + S_R_x * S_I_x

    return tf.reduce_mean(tf.square(f_R)) + tf.reduce_mean(tf.square(f_I))

def bc_loss(model, t_bc, x_lim):
    x_left = tf.fill([tf.shape(t_bc)[0], 1], -x_lim)
    x_right = tf.fill([tf.shape(t_bc)[0], 1], x_lim)

    inputs_left = tf.concat([x_left, t_bc], axis=1)
    inputs_right = tf.concat([x_right, t_bc], axis=1)

    S_left = model(inputs_left)
    S_right = model(inputs_right)

    target_S_R = 10.0

    loss_left = tf.reduce_mean(tf.square(S_left[:, 0:1] - target_S_R))
    loss_right = tf.reduce_mean(tf.square(S_right[:, 0:1] - target_S_R))

    return loss_left + loss_right

# ==========================================
# 3. Estructura Time-Marching (Marcha Temporal)
# ==========================================
t_max_sim = 8.0
n_windows = 4  # Dividimos los 10 segundos en 4 ventanas de 2.5s cada una
dt_window = t_max_sim / n_windows

n_colocacion = 30000  # Puntos por ventana
n_ic = 5000
n_bc = 5000
batch_size = 15000

pinn = RiccatiPINN()
pinn(tf.zeros((1, 2)))

# --- FUNCIÓN DE ENTRENAMIENTO COMPILADA POR VENTANA ---
@tf.function(jit_compile=True)
def train_step_window(model, optimizer, x_f_batch, t_f_batch, x_ic, t_ic, S_R_tgt, S_I_tgt, t_bc):
    with tf.GradientTape() as tape:
        loss_f = physics_loss(model, x_f_batch, t_f_batch)

        inputs_ic = tf.concat([x_ic, t_ic], axis=1)
        S_pred_ic = model(inputs_ic)
        loss_i = tf.reduce_mean(tf.square(S_pred_ic[:, 0:1] - S_R_tgt)) + \
                 tf.reduce_mean(tf.square(S_pred_ic[:, 1:2] - S_I_tgt))

        loss_b = bc_loss(model, t_bc, x_lim=3.0)

        total_loss = loss_f * 5.0 + loss_i * 10.0 + loss_b * 1.0

    grads = tape.gradient(total_loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return total_loss, loss_f, loss_i, loss_b

# Historiales globales concatenados
hist_total_all, hist_phys_all, hist_ic_all, hist_bc_all = [], [], [], []
trained_weights = []

for w in range(n_windows):
    t_start = w * dt_window
    t_end = (w + 1) * dt_window

    print(f"\n{'='*50}")
    print(f"🚀 ENTRENANDO VENTANA {w+1}/{n_windows}: t en [{t_start:.2f}, {t_end:.2f}]")
    print(f"{'='*50}")

    # Generar puntos específicos de la ventana en CPU
    x_f_np = np.random.uniform(-3, 3, (n_colocacion, 1)).astype(np.float32)
    t_f_np = np.random.uniform(t_start, t_end, (n_colocacion, 1)).astype(np.float32)
    x_ic_np = np.random.uniform(-3, 3, (n_ic, 1)).astype(np.float32)
    t_ic_np = np.full((n_ic, 1), t_start).astype(np.float32)
    t_bc_np = np.random.uniform(t_start, t_end, (n_bc, 1)).astype(np.float32)

    if w == 0:
        S_R_target_np = (0.5 * x_ic_np**2 + 0.25 * np.log(np.pi)).astype(np.float32)
        S_I_target_np = np.zeros_like(x_ic_np).astype(np.float32)
    else:
        inputs_prev = tf.concat([x_ic_np, t_ic_np], axis=1)
        S_prev_pred = pinn(inputs_prev).numpy()
        S_R_target_np = S_prev_pred[:, 0:1].astype(np.float32)
        S_I_target_np = S_prev_pred[:, 1:2].astype(np.float32)
        print(">> Condición de continuidad espacio-temporal inyectada con éxito.")

    x_ic_tf = tf.convert_to_tensor(x_ic_np, dtype=tf.float32)
    t_ic_tf = tf.convert_to_tensor(t_ic_np, dtype=tf.float32)
    S_R_tgt_tf = tf.convert_to_tensor(S_R_target_np, dtype=tf.float32)
    S_I_tgt_tf = tf.convert_to_tensor(S_I_target_np, dtype=tf.float32)
    t_bc_tf = tf.convert_to_tensor(t_bc_np, dtype=tf.float32)

    dataset_f = tf.data.Dataset.from_tensor_slices((x_f_np, t_f_np))\
        .shuffle(buffer_size=4096).batch(batch_size).cache().prefetch(tf.data.AUTOTUNE)

    epochs_window = 12000 # Épocas por tramo temporal
    lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate=0.005, decay_steps=4000, decay_rate=0.5, staircase=True)
    optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule)

    optimizer.build(pinn.trainable_variables)

    best_loss_w = float('inf')
    best_weights_w = None

    # El bucle de las épocas ahora está perfectamente anidado dentro de cada ventana
    for epoch in range(epochs_window + 1):
        epoch_tot, epoch_phys, epoch_ic, epoch_bc, steps = 0.0, 0.0, 0.0, 0.0, 0

        for x_f_batch, t_f_batch in dataset_f:
            tot_l, phys_l, ic_l, bc_l = train_step_window(
                pinn, optimizer, x_f_batch, t_f_batch, x_ic_tf, t_ic_tf, S_R_tgt_tf, S_I_tgt_tf, t_bc_tf)

            epoch_tot += tot_l.numpy()
            epoch_phys += phys_l.numpy()
            epoch_ic += (ic_l * 10.0).numpy() # Guardar pérdida ponderada de IC
            epoch_bc += bc_l.numpy()
            steps += 1

        epoch_tot /= steps
        epoch_phys /= steps
        epoch_ic /= steps
        epoch_bc /= steps

        hist_total_all.append(epoch_tot)
        hist_phys_all.append(epoch_phys)
        hist_ic_all.append(epoch_ic)
        hist_bc_all.append(epoch_bc)

        if epoch_tot < best_loss_w:
            best_loss_w = epoch_tot
            best_weights_w = pinn.get_weights()

        if epoch % 2000 == 0:
            print(f"  Época {epoch:5d}/{epochs_window} | Pérdida Total: {epoch_tot:.4e} | Física: {epoch_phys:.4e}")

    # Guardar pesos al finalizar la ventana actual
    pinn.set_weights(best_weights_w)
    trained_weights.append(best_weights_w)

print("\n¡Entrenamiento por Time-Marching completado exitosamente!")

# ==========================================
# 4. Gráfica del Historial de Pérdidas Concatenado
# ==========================================
fig_loss, ax_loss = plt.subplots(figsize=(10, 5))

ax_loss.plot(hist_total_all, label='Pérdida Total del Sistema', color='black', linewidth=1.5)
ax_loss.plot(hist_phys_all, label='Pérdida Física (Riccati)', color='blue', alpha=0.6)
ax_loss.plot(hist_ic_all, label='Pérdida Condición Inicial (x10)', color='red', alpha=0.6)
ax_loss.plot(hist_bc_all, label='Pérdida Frontera (BC)', color='magenta', alpha=0.6)

# Líneas divisorias de las ventanas de tiempo
for w in range(1, n_windows):
    ax_loss.axvline(x=w * (epochs_window + 1), color='red', linestyle='--', alpha=0.7, label='Corte de Ventana Causal' if w==1 else "")

ax_loss.set_yscale('log')
ax_loss.set_xlabel('Épocas Totales Acumuladas')
ax_loss.set_ylabel('Pérdida (Escala Logarítmica)')
ax_loss.set_title('Evolución de las Pérdidas bajo la Estrategia Causal Time-Marching')
ax_loss.legend(loc='upper right')
ax_loss.grid(True, which="both", ls="--", alpha=0.5)

plt.tight_layout()
plt.savefig('loss_pinn_duffing.png', dpi=300, bbox_inches='tight')
print("¡Gráfica de pérdidas guardada como 'loss_pinn_duffing.png'!")

# ==========================================
# 5. Evaluación y Ensamblaje Espacio-Temporal
# ==========================================
x_arr, t_arr, W_exact = get_ground_truth(t_max_sim, 100, 100)
X, T = np.meshgrid(x_arr, t_arr)

S_pred_global = np.zeros((len(X.flatten()), 2))
x_flat_np = X.flatten()[:, None]
t_flat_np = T.flatten()[:, None]

print("Ensamblando predicciones de las sub-ventanas temporales...")
for w in range(n_windows):
    t_start = w * dt_window
    t_end = (w + 1) * dt_window

    if w == n_windows - 1:
        mask = (t_flat_np >= t_start) & (t_flat_np <= t_max_sim)
    else:
        mask = (t_flat_np >= t_start) & (t_flat_np < t_end)

    mask = mask.flatten()

    if np.any(mask):
        pinn.set_weights(trained_weights[w])
        inputs_w = tf.concat([x_flat_np[mask], t_flat_np[mask]], axis=1)
        S_pred_global[mask] = pinn(inputs_w).numpy()

# Reconstrucción de la densidad de probabilidad W
S_R_pred = S_pred_global[:, 0].reshape(X.shape)
S_I_pred = S_pred_global[:, 1].reshape(X.shape)

psi_pinn = np.exp(-(S_R_pred + 1j * S_I_pred))
W_pinn = np.abs(psi_pinn)**2

dx = x_arr[1] - x_arr[0]
norm = np.sum(W_pinn, axis=1) * dx
W_pinn = W_pinn / norm[:, np.newaxis]

Error_W = np.abs(W_exact - W_pinn)

fig, axs = plt.subplots(1, 3, figsize=(18, 5))

c1 = axs[0].contourf(T, X, W_exact, levels=50, cmap='viridis')
axs[0].set_title(r'Original EDO ($W_{exact}$)')
axs[0].set_xlabel('Tiempo (t)')
axs[0].set_ylabel('Posición (x)')
fig.colorbar(c1, ax=axs[0])

c2 = axs[1].contourf(T, X, W_pinn, levels=50, cmap='viridis')
axs[1].set_title(r'Predicción PINN ($W_{PINN}$)')
axs[1].set_xlabel('Tiempo (t)')
fig.colorbar(c2, ax=axs[1])

c3 = axs[2].contourf(T, X, Error_W, levels=50, cmap='magma')
axs[2].set_title(r'Error Absoluto $|W_{exact} - W_{PINN}|$')
axs[2].set_xlabel('Tiempo (t)')
fig.colorbar(c3, ax=axs[2])

plt.tight_layout()
plt.savefig('resultado_pinn_duffing.png', dpi=300, bbox_inches='tight')
print("¡Gráfica espacio-temporal guardada como 'resultado_pinn_duffing.png'!")